<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/02-evals-basics/01-measuring-outputs.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 评测 I：度量一个概率性系统

**目标：** 养成那个能把 AI 工程师和 prompt 调参党区分开来的习惯：在你动手构建任何需要调优的东西之前，先在尽可能简单的系统上，给「好不好」这件事给出一个*数字*。

> **🔵 本篇的位置 ——** 评测是整门课的脊梁，所以它分**两轮**讲。这里是轻量的第一轮：在一个微小的任务上跑通 黄金集 → 指标 → 度量到的变化 这个循环，特意放在*前面*，让你在构建任何东西之前就先把习惯养成。**第 04 节（评测 II）** 才是针对真实 RAG 系统的严谨版本，包含 LLM-as-judge、人工校准，以及把回归评测接进 CI。两者之间，评测会在每一节里反复出现（RAG 检索、agent 轨迹、微调决策、生产监控）。刻意做得很短；深度在第 04 节。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

## 环境准备

每个 notebook 都是自包含的，接下来的两个单元格会把它从零搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的环境准备 helper，并拉入 `groq` 客户端——这个 notebook 唯一需要的依赖。
2. **载入你的 API key。** 免费 key 在 [console.groq.com](https://console.groq.com/) 领（无需信用卡）；在 Colab 里通过**钥匙图标** → **Add new secret** 添加，名称必须严格写成 `GROQ_API_KEY`，并打开 notebook access。本地运行则在环境变量里设置 `GROQ_API_KEY`。

（完整走查：[00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 载入 GROQ_API_KEY 并返回一个可直接使用的 Groq 客户端。
client, MODEL = setup()

> **🔵 撞上速率限制？换模型。** 评测类 notebook 是 token 消耗大户（每个 case 一次模型调用，而且反复跑），而 Groq 免费档对每分钟 token 数和每天 token 数都是**按模型**分别限制的。如果收到 `429`，就让 `setup()` 指向另一个模型；它有自己独立的一份额度：
> ```python
> client, MODEL = setup(model='openai/gpt-oss-20b')     # or 'qwen/qwen3.6-27b'
> ```
> 每个免费模型都有各自的限额；从 [Groq rate-limits 页面](https://console.groq.com/docs/rate-limits) 挑一个还有余量的。只是做*验证*的话，任何够用的对话模型都行。方法论才是重点，不是具体分数（也就是第 00 节那条「用便宜的开发，用目标模型评测」的规则）。这里避开 agentic 的 `groq/compound*` 模型：它们会联网浏览并执行代码，会破坏「答案得有据可依」这一课。

## 为什么这件事要排在所有事情前面

你刚刚学会了拿到结构化输出、调用工具（第 01 节）。现在你的本能是想做点更大的东西：RAG、agent。**请忍过一个 notebook 的时间。** 原因如下。

在普通软件里，「它能不能用？」有一个确定性的答案：测试要么过要么不过，编译器要么报错要么不报。LLM 系统打破了这个前提。同一个 prompt 这次给你好答案，下次给你错答案；一个改动让某个输入变好了，却悄悄弄坏另外三个；「我试的时候看起来是对的」正是坏系统上线的方式。你没办法*靠肉眼看*把一个 LLM 系统掰到能用。

唯一能让你有底气去构建、调优、改动任何东西的，是一个**能说明它是在变好还是变坏的数字**。这个数字就是 **eval（评测）**，也就是计划里称作*「那个分水岭」*的功夫。我们现在就把它装上，装在 第 01 节 那个微小的抽取任务上——那里机制一目了然——等你日后去调检索和 agent 时，它就已经是本能了，而恰恰在那两处，靠肉眼是最不灵的。

## 被测系统：第 01 节的抽取任务

`01-model-apis/01-structured-output` 里那个客服工单抽取器的精简版：输入自由文本邮件，输出结构化工单。这就是我们的被测系统，小到「*好*的输出长什么样」毫无歧义——而这正是你在学着度量时最想要的东西。

In [ ]:
import json, re
from groq import BadRequestError

TICKET_TOOL = {
    "type": "function",
    "function": {
        "name": "record_ticket",
        "description": "Record a structured support ticket from a customer email.",
        "parameters": {
            "type": "object",
            "properties": {
                "summary":  {"type": "string"},
                "priority": {"type": "string", "enum": ["low", "medium", "high", "urgent"]},
                "category": {"type": "string", "enum": ["billing", "bug", "account", "other"]},
            },
            "required": ["summary", "priority", "category"],
        },
    },
}

def extract_ticket(email, system=None, tries=3):
    # 强制指定 tool_choice 会让模型把这次调用当作文本吐出来、再由 API 解析；当
    # 这个解析失败时，Groq 会抛 400 tool_use_failed，而不是返回一条消息。
    # 它是间歇性的，所以放弃前先重试几次（第 08 节会深入讲这个
    # 可靠性模式）。
    messages = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": f"Extract the ticket:\n\n{email}"}]
    for _ in range(tries):
        try:
            resp = client.chat.completions.create(
                model=MODEL, max_tokens=200, tools=[TICKET_TOOL],
                tool_choice={"type": "function", "function": {"name": "record_ticket"}},
                messages=messages,
            )
            return json.loads(resp.choices[0].message.tool_calls[0].function.arguments)
        except BadRequestError as e:
            if 'tool_use_failed' in str(e):
                continue
            raise
    return {"summary": "", "priority": None, "category": None}  # 哨兵值：算作一个失败的 case

# 稳定的私有别名，这样下面的 v2 变体即使在之后换掉了 `extract_ticket`
# 这个名字，也仍能调用基础抽取器（避免重跑那个单元格时发生递归）。
_extract_base = extract_ticket

print(extract_ticket("Hi, I was charged twice for my subscription this month. Please refund. Urgent!"))

## 第 1 步：黄金集，你拿来对照的东西

评测需要*基准真值*（ground truth）：输入要配上「正确的输出应该是什么」。这就是**黄金集**：手写、可信、量小。十个好 case 胜过一千个没有标注的。既要覆盖容易的路径，*也*要覆盖你担心的那些 case（在这里就是：优先级模棱两可的、分类可能两可的、写得很简短的邮件）。

In [ ]:
# 每个 case：输入的邮件 + 一次正确抽取必须答对的字段。
# 我们不钉死 summary 的具体措辞（有很多种写法都行）—— 只钉死
# 机器可校验的字段。选对断言什么，是评测功夫的一半。
GOLDEN = [
    {"email": "I was double-charged for my plan this month, need a refund asap.",
     "priority": "urgent", "category": "billing"},
    {"email": "The export button throws a 500 error every time I click it.",
     "priority": "high", "category": "bug"},
    {"email": "How do I change the email address on my account?",
     "priority": "low", "category": "account"},
    {"email": "Your product is fine I guess. Just leaving feedback.",
     "priority": "low", "category": "other"},
    {"email": "URGENT: production is down, your API returns 503 for all calls!!",
     "priority": "urgent", "category": "bug"},
    {"email": "I'd like to upgrade to the annual plan — what's the price?",
     "priority": "low", "category": "billing"},
]
print(len(GOLDEN), "golden cases")

## 第 2 步：指标 —— 把对/错变成一个数字

对结构化输出而言，指标简单且确定，还不需要 LLM judge（那是第 04 节，用于输出是自由文本、*没法*做字符串匹配的场景）。这里：

- **Schema 合法性** —— 它到底有没有按正确的形状返回？（一次「成功」但返回垃圾的调用，就是失败）
- **字段准确率** —— 对我们钉死的那些字段，它答对了吗？

跑一遍，同时读按 case 的表格*和*汇总值。汇总值是你长期追踪的东西；按 case 的那些行是你去 debug 的地方。

In [ ]:
def valid_schema(t):
    return (isinstance(t.get("summary"), str)
            and t.get("priority") in {"low","medium","high","urgent"}
            and t.get("category") in {"billing","bug","account","other"})

def evaluate(golden):
    rows, correct_fields, total_fields, valid = [], 0, 0, 0
    for case in golden:
        got = extract_ticket(case["email"])
        ok_schema = valid_schema(got)
        valid += ok_schema
        # 在钉死的字段上的字段准确率
        p_ok = got.get("priority") == case["priority"]
        c_ok = got.get("category") == case["category"]
        correct_fields += p_ok + c_ok; total_fields += 2
        rows.append((case["email"][:42], got.get("priority"), p_ok, got.get("category"), c_ok))
        print(f"  pri={str(got.get('priority')):8}{'ok' if p_ok else 'XX'}  "
              f"cat={str(got.get('category')):8}{'ok' if c_ok else 'XX'}  | {case['email'][:40]}")
    print("-"*72)
    print(f"schema valid : {valid}/{len(golden)}")
    print(f"field accuracy: {correct_fields}/{total_fields} = {correct_fields/total_fields:.0%}")
    return correct_fields/total_fields

score = evaluate(GOLDEN)

## 第 3 步：现在一个改动是一个决策，而不是一次猜测

要点全在这里。假设你想通过往 prompt 里加一条规则来「改进」抽取器。没有评测时，你试一下、看一眼某个输出，然后凭感觉上线。有了评测，你会**把两者都度量一遍再对比**：这个改动要么让数字动了，要么没动。

In [ ]:
# 一次 prompt 微调：加入明确的优先级指引。它真的有用吗？去度量它。
PRIORITY_GUIDE = ("Priority guide: outages/data-loss/double-charges = urgent; "
                  "broken features = high; questions and feedback = low.")

def extract_ticket_v2(email):
    # 通过那个稳定的别名复用基础抽取器（连同它的重试逻辑），只是多加一段
    # system prompt —— 所以 v2 相对 v1 是真正的单变量改动。
    return _extract_base(email, system=PRIORITY_GUIDE)

# 换掉测试骨架使用的抽取器，重跑，再和第 2 步的 `score` 对比。
globals()["extract_ticket"] = extract_ticket_v2
print("--- v2 (with priority guidance):")
score_v2 = evaluate(GOLDEN)
print(f"\nv1 = {score:.0%}  ->  v2 = {score_v2:.0%}  "
      f"({'better' if score_v2>score else 'no better — do not ship the extra tokens'})")

那次对比——*v1 与 v2，用同一个数字，在同一批 case 上*——就是这门课余下部分以及你整个职业生涯里每一次改进的最小单元。你刚刚在一个玩具任务上做的，正是第 04 节对 RAG 系统严格去做的事（黄金集、LLM-as-judge，以及把评测接进 CI）、第 05 节对 agent 轨迹做的事，以及第 08 节从生产 trace 回灌的事。

> **⭐ 关键收获 ——** 在你调*任何东西*之前（prompt、chunk 大小、retriever、模型），先问「我的黄金集是什么，数字是多少？」如果你没有，那你不是在改进系统，你只是在改动它。

从这里往后的每一节，都默认你已经这样思考了。

## 练习

1. **先弄坏，再抓住它。** 往 v2 的 system prompt 里加一条故意写坏的规则（例如「一切都是低优先级」）。确认评测分数*下降*了。你刚刚证明了你的评测能发现回归——而这正是它存在的全部理由。
2. **补上你害怕的那些 case。** 再写 4 个针对歧义场景的黄金 case：一封既像计费又像 bug 的邮件、一封阴阳怪气的、一句非英文的话、一条几乎为空的留言。抽取器会答错哪些？每一个失败都值得留成一个 case。
3. **加一个指标。** 扩展 `evaluate`，用一个便宜的代理指标也度量 summary 的质量：例如 summary 长度是否在 5 到 100 个字符之间，以及它是否和邮件共享一个关键词。留意在哪些地方确定性代理就够了，在哪些地方你最终会想要 LLM judge（第 04 节）。
4. **评测的成本。** 测试骨架每个 case 要发一次模型调用。你会怎样让黄金集快到、便宜到能在*每一次*改动上都跑一遍？（这个问题正是第 04 节的 regression-in-CI notebook 所回答的。）